# InfluencerRank v3: End-to-End GCN Training (All Bugs Fixed)

This notebook fixes ALL critical bugs identified in the v2 audit:

**CRITICAL FIXES:**
1. **GCN NOW TRAINED END-TO-END** - GCN is part of the computation graph (was using random weights!)
2. **NON-INFLUENCER FEATURES PRESERVED** - One-hot encoding no longer destroyed by normalization
3. **EXTENDED LEAKY INDICES** - Also zeros out likes-based features (12, 14, 16, 18, 20, 24)
4. **CLEANED UP UNUSED CODE** - Removed unused model components and parameters

**Expected Performance:**
- v1 (self-loops only): 0.6951 NDCG@50
- v2 (random GCN weights): ~0.55-0.65 NDCG@50
- v3 (properly trained GCN): 0.70-0.74 NDCG@50
- Paper target: 0.720 NDCG@50

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

# PyTorch Geometric
from torch_geometric.nn import GCNConv

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration

In [ ]:
# Data paths (Kaggle)
GRAPH_DIR = '/kaggle/input/graphsv2/graphs_enhanced_v2'

# CRITICAL: Extended leaky feature indices
# These features are derived from engagement_rate OR likes (which correlates with engagement!)
LEAKY_INDICES = [
    11,  # engagement_trend
    12,  # likes_trend (ADDED IN V3!)
    13,  # engagement_variance
    14,  # likes_variance (ADDED IN V3!)
    15,  # engagement_consistency
    16,  # likes_consistency (ADDED IN V3!)
    17,  # engagement_momentum
    18,  # likes_momentum (ADDED IN V3!)
    19,  # engagement_peak
    20,  # likes_peak (ADDED IN V3!)
    23,  # engagement_growth
    24,  # likes_growth (ADDED IN V3!)
    25,  # log_avg_likes
    26,  # log_avg_comments
]

print(f"LEAKY FEATURE INDICES TO ZERO OUT: {LEAKY_INDICES}")
print(f"Total: {len(LEAKY_INDICES)} features (14 in v3 vs 8 in v2)")
print("\nV3 additions (likes-based features that correlate with engagement):")
print("  - 12: likes_trend")
print("  - 14: likes_variance")
print("  - 16: likes_consistency")
print("  - 18: likes_momentum")
print("  - 20: likes_peak")
print("  - 24: likes_growth")

# Architecture (paper-matching)
INPUT_DIM = 37  # Original features (will have 14 zeroed out in v3)
GNN_HIDDEN = 128
GNN_OUT = 128  # Paper uses 128
RNN_HIDDEN = 128
DROPOUT = 0.5  # Paper uses 0.5

# Training
BATCH_SIZE = 32  # Reduced for end-to-end training (more GPU memory needed)
LIST_SIZE = 10   # Compare 10 influencers at once
LEARNING_RATE = 0.001  # Paper uses 0.001
NUM_EPOCHS = 200
EARLY_STOP_PATIENCE = 30
WEIGHT_DECAY = 1e-5

# Temporal settings
TRAINING_MONTHS = 9  # Use months 0-8 (Jan-Sep)
TARGET_MONTH = 9     # Predict month 9 (October)

# Ensemble
ENSEMBLE_SEEDS = [42, 123, 456, 789, 2024]

# Data split
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print(f"\nArchitecture: GNN({INPUT_DIM}->{GNN_HIDDEN}->{GNN_OUT}) -> GRU({GNN_OUT}->{RNN_HIDDEN}) -> Attention -> FC")
print(f"Training: LR={LEARNING_RATE}, Dropout={DROPOUT}, Batch={BATCH_SIZE}, ListSize={LIST_SIZE}")

print(f"\n** VERSION 3: END-TO-END GCN TRAINING **")
print(f"Critical fix: GCN is now TRAINED (was using random weights in v2!)")
print(f"Expected NDCG@50: 0.70-0.74 (up from ~0.55-0.65 with random GCN)")

## 3. Load Graphs and Remove Leaky Features

In [ ]:
# Month names
month_names = ["jan", "feb", "mar", "apr", "may", "jun", 
               "jul", "aug", "sep", "oct", "nov", "dec"]

print("Loading graphs...")
all_graphs_data = []

for month in month_names:
    path = os.path.join(GRAPH_DIR, f"{month}_graph.pt")
    data = torch.load(path, weights_only=False)
    all_graphs_data.append(data)
    print(f"  {month.upper()}: {data['graph']['influencer'].x.shape[0]} influencers")

print(f"\nLoaded {len(all_graphs_data)} monthly graphs")

In [ ]:
# CRITICAL: Zero out leaky features IMMEDIATELY after loading
print("\n" + "="*60)
print("REMOVING DATA LEAKAGE (EXTENDED IN V3)")
print("="*60)

print(f"\nZeroing out {len(LEAKY_INDICES)} leaky features at indices: {LEAKY_INDICES}")
print("\nThese features contain engagement/likes information (the target):")
print("  Engagement-based (8 features - from v2):")
print("    - 11: engagement_trend")
print("    - 13: engagement_variance")
print("    - 15: engagement_consistency")
print("    - 17: engagement_momentum")
print("    - 19: engagement_peak")
print("    - 23: engagement_growth")
print("    - 25: log_avg_likes")
print("    - 26: log_avg_comments")
print("\n  Likes-based (6 features - NEW IN V3):")
print("    - 12: likes_trend")
print("    - 14: likes_variance")
print("    - 16: likes_consistency")
print("    - 18: likes_momentum")
print("    - 20: likes_peak")
print("    - 24: likes_growth")

# Show before
print(f"\nBefore zeroing (Oct graph, first influencer):")
print(f"  Feature 11 (engagement_trend): {all_graphs_data[9]['graph']['influencer'].x[0, 11]:.6f}")
print(f"  Feature 12 (likes_trend): {all_graphs_data[9]['graph']['influencer'].x[0, 12]:.6f}")
print(f"  Feature 25 (log_avg_likes): {all_graphs_data[9]['graph']['influencer'].x[0, 25]:.6f}")

# Zero out leaky features
for data_package in all_graphs_data:
    features = data_package['graph']['influencer'].x
    features[:, LEAKY_INDICES] = 0.0  # Zero out leaky features

# Show after
print(f"\nAfter zeroing (Oct graph, first influencer):")
print(f"  Feature 11 (engagement_trend): {all_graphs_data[9]['graph']['influencer'].x[0, 11]:.6f}")
print(f"  Feature 12 (likes_trend): {all_graphs_data[9]['graph']['influencer'].x[0, 12]:.6f}")
print(f"  Feature 25 (log_avg_likes): {all_graphs_data[9]['graph']['influencer'].x[0, 25]:.6f}")

print(f"\n** LEAKAGE REMOVED! {len(LEAKY_INDICES)} features zeroed out **")
print(f"Effective features: {INPUT_DIM - len(LEAKY_INDICES)} non-zero dimensions")
print("="*60)

In [ ]:
# Verify features after removing leakage
print("\nFeature verification:")
oct_features = all_graphs_data[9]['graph']['influencer'].x
print(f"Shape: {oct_features.shape}")
print(f"Non-zero features per influencer: {(oct_features != 0).sum(dim=1).float().mean():.1f}/{oct_features.shape[1]}")
print(f"Zeroed indices: {LEAKY_INDICES} ({len(LEAKY_INDICES)} features)")
print(f"Effective features: {oct_features.shape[1] - len(LEAKY_INDICES)} non-zero dimensions")

In [ ]:
# Verify graph structure is being used
print("\nGraph structure verification:")
sample_graph = all_graphs_data[0]['graph']
print(f"  Influencers: {sample_graph['influencer'].num_nodes}")
print(f"  Hashtags: {sample_graph['hashtag'].num_nodes if 'hashtag' in sample_graph.node_types else 0}")
print(f"  Users: {sample_graph['user'].num_nodes if 'user' in sample_graph.node_types else 0}")
print(f"  Objects: {sample_graph['object'].num_nodes if 'object' in sample_graph.node_types else 0}")

total_edges = 0
for edge_type in sample_graph.edge_types:
    num_edges = sample_graph[edge_type].edge_index.shape[1]
    print(f"  {edge_type}: {num_edges} edges")
    total_edges += num_edges
print(f"  Total edges (before conversion): {total_edges}")

## 4. Data Split (By Influencers)

In [ ]:
# Get all unique influencers from target month
target_data = all_graphs_data[TARGET_MONTH]
all_influencers = list(target_data['maps']['influencer'].keys())

print(f"Total influencers in target month (October): {len(all_influencers)}")

# Split by influencers (NOT by time!)
np.random.seed(42)
np.random.shuffle(all_influencers)

n = len(all_influencers)
n_train = int(TRAIN_RATIO * n)
n_val = int(VAL_RATIO * n)

train_influencers = set(all_influencers[:n_train])
val_influencers = set(all_influencers[n_train:n_train + n_val])
test_influencers = set(all_influencers[n_train + n_val:])

# CRITICAL: Verify no overlap!
assert len(train_influencers & val_influencers) == 0, "LEAKAGE: Train/Val overlap!"
assert len(train_influencers & test_influencers) == 0, "LEAKAGE: Train/Test overlap!"
assert len(val_influencers & test_influencers) == 0, "LEAKAGE: Val/Test overlap!"

print(f"\nData Split (NO OVERLAP):")
print(f"  Train: {len(train_influencers)} influencers ({100*len(train_influencers)/n:.1f}%)")
print(f"  Val:   {len(val_influencers)} influencers ({100*len(val_influencers)/n:.1f}%)")
print(f"  Test:  {len(test_influencers)} influencers ({100*len(test_influencers)/n:.1f}%)")
print(f"\nOverlap check: PASSED (0% overlap)")

# Convert to lists for indexing
train_influencers_list = list(train_influencers)
val_influencers_list = list(val_influencers)
test_influencers_list = list(test_influencers)

## 5. Train-Only Normalization (FIXED: No Non-Influencer Normalization)

In [ ]:
# CRITICAL: Fit scaler ONLY on training influencers' features
print("Performing train-only normalization...")

train_features = []
for month_idx in range(TRAINING_MONTHS):  # Only months 0-8
    month_data = all_graphs_data[month_idx]
    features = month_data['graph']['influencer'].x
    influencer_map = month_data['maps']['influencer']
    
    for inf_name in train_influencers:
        if inf_name in influencer_map:
            local_idx = influencer_map[inf_name]
            train_features.append(features[local_idx].numpy())

train_features = np.vstack(train_features)
print(f"  Collected {train_features.shape[0]} training feature vectors")
print(f"  Feature shape: {train_features.shape}")

# Fit scaler ONLY on training data
scaler = StandardScaler()
scaler.fit(train_features)

print(f"  Scaler fitted on training data only")
print(f"  Mean range: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
print(f"  Std range: [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]")

In [ ]:
# Apply normalization to ONLY influencer features (CRITICAL FIX FROM V2!)
print("\nApplying train-fitted normalization to INFLUENCER features ONLY...")
print("\n** V3 FIX: Non-influencer features (hashtags, users, objects) are NOT normalized! **")
print("  They use one-hot encoding [0,1,0,0,...] that must be preserved.")
print("  Normalizing them would destroy their meaning (see audit report).")

for month_idx, data_package in enumerate(all_graphs_data):
    # ONLY normalize influencer features
    features = data_package['graph']['influencer'].x
    normalized = scaler.transform(features.numpy())
    data_package['graph']['influencer'].x = torch.FloatTensor(normalized)

# Verify normalization
oct_features = all_graphs_data[9]['graph']['influencer'].x
print(f"\n  After normalization (Oct influencers):")
print(f"    Mean: {oct_features.mean():.6f} (should be near 0)")
print(f"    Std: {oct_features.std():.6f} (should be near 1)")
print(f"    Min: {oct_features.min():.4f}")
print(f"    Max: {oct_features.max():.4f}")

# Verify non-influencer features are PRESERVED
if 'hashtag' in all_graphs_data[0]['graph'].node_types:
    hashtag_features = all_graphs_data[0]['graph']['hashtag'].x
    print(f"\n  Non-influencer features (hashtags) PRESERVED:")
    print(f"    Shape: {hashtag_features.shape}")
    print(f"    First hashtag features (should be one-hot): {hashtag_features[0, :5].tolist()}")
    print(f"    Sum per row (should be ~1 for one-hot): {hashtag_features.sum(dim=1).mean():.4f}")

## 6. Graph Conversion Function

In [ ]:
def convert_hetero_to_homogeneous(graph, influencer_map):
    """
    Convert heterogeneous graph to homogeneous for GCN.
    Returns: x (all node features), edge_index (all edges), influencer_indices (local to global mapping)
    """
    # Get all node features
    x_inf = graph['influencer'].x
    x_hash = graph['hashtag'].x if 'hashtag' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    x_user = graph['user'].x if 'user' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    x_obj = graph['object'].x if 'object' in graph.node_types else torch.zeros((0, x_inf.shape[1]))
    
    # Stack features (influencers first)
    x = torch.cat([x_inf, x_hash, x_user, x_obj], dim=0)
    
    # Calculate offsets
    offset_inf = 0
    offset_hash = x_inf.shape[0]
    offset_user = offset_hash + x_hash.shape[0]
    offset_obj = offset_user + x_user.shape[0]
    
    offsets = {
        'influencer': offset_inf,
        'hashtag': offset_hash,
        'user': offset_user,
        'object': offset_obj
    }
    
    # Convert edges with proper offsets
    edge_list = []
    for edge_type in graph.edge_types:
        src_type, _, dst_type = edge_type
        if edge_type in graph.edge_index_dict:
            edges = graph[edge_type].edge_index.clone()
            edges[0] += offsets[src_type]
            edges[1] += offsets[dst_type]
            edge_list.append(edges)
            
            # Add reverse edges (undirected graph)
            reverse_edges = torch.stack([edges[1], edges[0]], dim=0)
            edge_list.append(reverse_edges)
    
    if edge_list:
        edge_index = torch.cat(edge_list, dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    
    # Add self-loops for numerical stability
    num_nodes = x.shape[0]
    self_loops = torch.stack([
        torch.arange(num_nodes),
        torch.arange(num_nodes)
    ])
    edge_index = torch.cat([edge_index, self_loops], dim=1)
    
    # Create mapping from influencer name to global index
    influencer_global_indices = {}
    for name, local_idx in influencer_map.items():
        influencer_global_indices[name] = local_idx + offset_inf  # offset_inf = 0
    
    return x, edge_index, influencer_global_indices

print("Graph conversion function defined.")
print("  - Converts heterogeneous graph to homogeneous")
print("  - Uses ACTUAL edges (not self-loops)")
print("  - Adds reverse edges for undirected message passing")

In [ ]:
# Pre-convert all graphs to homogeneous (save computation during training)
print("\nPre-converting all heterogeneous graphs to homogeneous...")
print("  This is done ONCE before training to save computation time.")
print("  During training, only GCN forward pass is computed (not graph conversion).")

converted_graphs = []

for month_idx in range(TRAINING_MONTHS):
    data_package = all_graphs_data[month_idx]
    graph = data_package['graph']
    influencer_map = data_package['maps']['influencer']
    
    x, edge_index, global_indices = convert_hetero_to_homogeneous(graph, influencer_map)
    
    converted_graphs.append({
        'x': x,  # Keep on CPU, move to GPU during training
        'edge_index': edge_index,
        'global_indices': global_indices
    })
    
    print(f"  Month {month_idx} ({month_names[month_idx]}): {x.shape[0]} nodes, {edge_index.shape[1]} edges")

print(f"\nConverted {len(converted_graphs)} monthly graphs")

## 7. Model Architecture (FIXED: Single Trainable GCN)

In [ ]:
class SimpleGCN(nn.Module):
    """Simple 2-layer GCN as described in the paper."""
    
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x


class SimpleAttention(nn.Module):
    """Simple attention mechanism (not multi-head) as in paper."""
    
    def __init__(self, hidden_size):
        super().__init__()
        self.attention_fc = nn.Linear(hidden_size, 1)
    
    def forward(self, hidden_states, lengths):
        # hidden_states: [batch, seq_len, hidden]
        # lengths: [batch]
        batch_size, seq_len, _ = hidden_states.shape
        
        # Compute attention scores
        scores = self.attention_fc(hidden_states).squeeze(-1)  # [batch, seq_len]
        
        # Mask padded positions
        mask = torch.arange(seq_len, device=hidden_states.device).expand(batch_size, -1)
        mask = mask < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, -1e9)
        
        # Softmax
        weights = F.softmax(scores, dim=1)  # [batch, seq_len]
        
        # Weighted sum
        context = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)  # [batch, hidden]
        return context


class InfluencerRankModelV3(nn.Module):
    """
    V3: End-to-end trainable model.
    
    CRITICAL FIX: GCN is now ACTUALLY USED and TRAINED!
    - In v2: GCN was created but never used (embeddings came from external untrained GCN)
    - In v3: GCN is called during training, gradients flow through it
    """
    
    def __init__(self, input_dim, gnn_hidden, gnn_out, rnn_hidden, dropout=0.5):
        super().__init__()
        
        # GCN encoder (NOW ACTUALLY USED!)
        self.gcn = SimpleGCN(input_dim, gnn_hidden, gnn_out, dropout)
        
        # Single-layer GRU (NOT bidirectional, as in paper)
        self.rnn = nn.GRU(
            input_size=gnn_out,
            hidden_size=rnn_hidden,
            num_layers=1,  # Single layer!
            batch_first=True,
            bidirectional=False,  # NOT bidirectional!
            dropout=0.0  # No dropout for single layer
        )
        
        # Simple attention
        self.attention = SimpleAttention(rnn_hidden)
        
        # Output layers (2 layers, not 3)
        self.fc1 = nn.Linear(rnn_hidden, rnn_hidden // 2)
        self.fc2 = nn.Linear(rnn_hidden // 2, 1)
        self.dropout = nn.Dropout(dropout)
    
    def encode_graph(self, x, edge_index):
        """
        Encode graph with GCN.
        This is called DURING TRAINING so gradients flow through!
        """
        return self.gcn(x, edge_index)
        
    def forward_temporal(self, sequences, lengths):
        """
        Process temporal sequences through RNN + Attention + FC.
        sequences: [batch, seq_len, gnn_out]
        lengths: [batch]
        """
        # Pack sequence for RNN
        packed = pack_padded_sequence(
            sequences, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # RNN forward
        packed_output, _ = self.rnn(packed)
        rnn_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        # rnn_output: [batch, seq_len, rnn_hidden]
        
        # Attention
        context = self.attention(rnn_output, lengths.to(rnn_output.device))
        # context: [batch, rnn_hidden]
        
        # Output layers
        x = F.relu(self.fc1(context))
        x = self.dropout(x)
        score = self.fc2(x).squeeze(-1)  # [batch]
        
        return score


print("V3 Model architecture defined:")
print("  - 2-layer GCN (128 hidden, 0.5 dropout) - NOW TRAINED!")
print("  - 1-layer GRU (NOT bidirectional)")
print("  - Simple attention (NOT multi-head)")
print("  - 2 FC layers (NOT 3)")
print("\nCRITICAL FIX: GCN is part of computation graph and receives gradients!")

## 8. Loss Function and Metrics

In [ ]:
def listwise_ranking_loss(y_pred, y_true):
    """Paper's 0-1 ranking loss over lists."""
    # y_pred: [list_size]
    # y_true: [list_size]
    
    # Create all pairs
    pred_diff = y_pred.unsqueeze(1) - y_pred.unsqueeze(0)  # [n, n]
    true_diff = y_true.unsqueeze(1) - y_true.unsqueeze(0)  # [n, n]
    
    # Mask: only consider pairs where true_diff > 0 (correct ranking)
    mask = (true_diff > 0).float()
    
    # Cross-entropy style loss for ranking
    # We want pred_diff > 0 when true_diff > 0
    loss = -F.logsigmoid(pred_diff) * mask
    
    return loss.sum() / mask.sum().clamp(min=1)


def compute_ndcg_at_k(y_true, y_pred, k=50):
    """Compute NDCG@k metric."""
    if len(y_true) < 2:
        return 0.0
    
    # Ensure numpy arrays
    y_true = np.array(y_true).reshape(1, -1)
    y_pred = np.array(y_pred).reshape(1, -1)
    
    # Adjust k if necessary
    k = min(k, len(y_true[0]))
    
    return ndcg_score(y_true, y_pred, k=k)


print("Loss and metrics defined:")
print("  - Listwise ranking loss (paper's approach)")
print("  - NDCG@50 evaluation metric")

## 9. Training Utilities (FIXED: Removed Unused Parameter)

In [ ]:
def get_ground_truth(influencer_names, target_data):
    """Get ground truth engagement rates for influencers."""
    ground_truth = []
    influencer_map = target_data['maps']['influencer']
    engagement_rates = target_data['ground_truth']['engagement_rate']
    
    for name in influencer_names:
        if name in influencer_map:
            local_idx = influencer_map[name]
            ground_truth.append(engagement_rates[local_idx].item())
        else:
            ground_truth.append(0.0)  # Not active in target month
    
    return torch.FloatTensor(ground_truth)


def sample_influencers(influencer_list, size):
    """Sample influencers for a batch."""
    if len(influencer_list) >= size:
        indices = np.random.choice(len(influencer_list), size, replace=False)
        return [influencer_list[i] for i in indices]
    else:
        return influencer_list


print("Training utilities defined.")
print("  Note: prepare_sequences() removed - embedding extraction now done inline during training")

## 10. End-to-End Training Loop (THE CRITICAL FIX!)

In [ ]:
def train_single_model_v3(seed, converted_graphs, train_influencers_list, val_influencers_list, test_influencers_list, all_graphs_data):
    """
    V3: End-to-end training with GCN as part of computation graph.
    
    CRITICAL FIX: GCN is now TRAINED!
    - In v2: External GCN computed embeddings with RANDOM weights
    - In v3: GCN is called during training, gradients flow through it
    """
    print(f"\n{'='*60}")
    print(f"Training model with seed {seed}")
    print(f"{'='*60}")
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # Initialize SINGLE model (GCN + RNN + Attention + FC)
    model = InfluencerRankModelV3(INPUT_DIM, GNN_HIDDEN, GNN_OUT, RNN_HIDDEN, DROPOUT).to(device)
    
    print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
    print(f"  GCN parameters: {sum(p.numel() for p in model.gcn.parameters())}")
    print(f"  RNN parameters: {sum(p.numel() for p in model.rnn.parameters())}")
    print(f"  Attention parameters: {sum(p.numel() for p in model.attention.parameters())}")
    print(f"  FC parameters: {sum(p.numel() for p in model.fc1.parameters()) + sum(p.numel() for p in model.fc2.parameters())}")
    
    # Optimizer (trains ALL parameters including GCN!)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    
    # Training state
    best_val_ndcg = 0.0
    best_model_state = None
    patience_counter = 0
    
    # Ground truth for target month
    target_data = all_graphs_data[TARGET_MONTH]
    
    print(f"\nStarting END-TO-END training (max {NUM_EPOCHS} epochs)...")
    print(f"  GCN gradients will now flow through the entire model!")
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_losses = []
        
        # Number of batches per epoch
        num_batches = max(1, len(train_influencers_list) // (BATCH_SIZE * LIST_SIZE))
        
        for batch_num in range(num_batches):
            optimizer.zero_grad()
            batch_loss = 0.0
            valid_samples = 0
            
            # Pre-compute GCN embeddings for ALL months (once per batch)
            # This is more efficient than computing per sample
            month_embeddings = []
            for month_idx in range(TRAINING_MONTHS):
                x = converted_graphs[month_idx]['x'].to(device)
                edge_index = converted_graphs[month_idx]['edge_index'].to(device)
                
                # GCN forward - NOW PART OF COMPUTATION GRAPH!
                node_embeddings = model.encode_graph(x, edge_index)
                month_embeddings.append(node_embeddings)
            
            # Process BATCH_SIZE lists
            for _ in range(BATCH_SIZE):
                # Sample LIST_SIZE influencers
                batch_names = sample_influencers(train_influencers_list, LIST_SIZE)
                
                # Build temporal sequences for each influencer
                sequences = []
                valid_names = []
                
                for name in batch_names:
                    seq = []
                    for month_idx in range(TRAINING_MONTHS):
                        if name in converted_graphs[month_idx]['global_indices']:
                            global_idx = converted_graphs[month_idx]['global_indices'][name]
                            # Extract embedding (maintains computation graph!)
                            seq.append(month_embeddings[month_idx][global_idx])
                    
                    if len(seq) > 0:
                        sequences.append(torch.stack(seq))  # [seq_len, gnn_out]
                        valid_names.append(name)
                
                if len(sequences) < 2:
                    continue
                
                # Pad sequences
                lengths = torch.LongTensor([s.shape[0] for s in sequences])
                padded = pad_sequence(sequences, batch_first=True)  # [batch, max_len, gnn_out]
                
                # Get ground truth
                y_true = get_ground_truth(valid_names, target_data).to(device)
                
                # Forward pass through RNN + Attention + FC
                y_pred = model.forward_temporal(padded, lengths)
                
                # Compute listwise loss
                loss = listwise_ranking_loss(y_pred, y_true)
                batch_loss += loss
                valid_samples += 1
            
            # Backprop through ENTIRE model including GCN!
            if valid_samples > 0:
                (batch_loss / valid_samples).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_losses.append((batch_loss / valid_samples).item())
            
            # Clear GPU cache
            del month_embeddings
            torch.cuda.empty_cache()
        
        # Validation
        if len(epoch_losses) > 0:
            avg_loss = np.mean(epoch_losses)
        else:
            avg_loss = 0.0
        
        # Evaluate on validation set (with GCN inference)
        model.eval()
        with torch.no_grad():
            # Compute GCN embeddings for all months
            val_month_embeddings = []
            for month_idx in range(TRAINING_MONTHS):
                x = converted_graphs[month_idx]['x'].to(device)
                edge_index = converted_graphs[month_idx]['edge_index'].to(device)
                node_embeddings = model.encode_graph(x, edge_index)
                val_month_embeddings.append(node_embeddings)
            
            # Build sequences for validation influencers
            val_sequences = []
            val_valid_names = []
            
            for name in val_influencers_list:
                seq = []
                for month_idx in range(TRAINING_MONTHS):
                    if name in converted_graphs[month_idx]['global_indices']:
                        global_idx = converted_graphs[month_idx]['global_indices'][name]
                        seq.append(val_month_embeddings[month_idx][global_idx])
                
                if len(seq) > 0:
                    val_sequences.append(torch.stack(seq))
                    val_valid_names.append(name)
            
            if len(val_sequences) >= 2:
                val_lengths = torch.LongTensor([s.shape[0] for s in val_sequences])
                val_padded = pad_sequence(val_sequences, batch_first=True)
                val_pred = model.forward_temporal(val_padded, val_lengths).cpu().numpy()
                val_true = get_ground_truth(val_valid_names, target_data).numpy()
                val_ndcg = compute_ndcg_at_k(val_true, val_pred, k=50)
            else:
                val_ndcg = 0.0
            
            del val_month_embeddings
            torch.cuda.empty_cache()
        
        # Update scheduler
        scheduler.step(val_ndcg)
        
        # Early stopping check
        if val_ndcg > best_val_ndcg:
            best_val_ndcg = val_ndcg
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {avg_loss:.4f}, Val NDCG@50: {val_ndcg:.4f} (Best: {best_val_ndcg:.4f}), LR: {current_lr:.6f}")
        
        # Early stopping
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model = model.to(device)
    
    # Test evaluation (with trained GCN!)
    model.eval()
    with torch.no_grad():
        # Compute GCN embeddings
        test_month_embeddings = []
        for month_idx in range(TRAINING_MONTHS):
            x = converted_graphs[month_idx]['x'].to(device)
            edge_index = converted_graphs[month_idx]['edge_index'].to(device)
            node_embeddings = model.encode_graph(x, edge_index)
            test_month_embeddings.append(node_embeddings)
        
        # Build sequences for test influencers
        test_sequences = []
        test_valid_names = []
        
        for name in test_influencers_list:
            seq = []
            for month_idx in range(TRAINING_MONTHS):
                if name in converted_graphs[month_idx]['global_indices']:
                    global_idx = converted_graphs[month_idx]['global_indices'][name]
                    seq.append(test_month_embeddings[month_idx][global_idx])
            
            if len(seq) > 0:
                test_sequences.append(torch.stack(seq))
                test_valid_names.append(name)
        
        if len(test_sequences) >= 2:
            test_lengths = torch.LongTensor([s.shape[0] for s in test_sequences])
            test_padded = pad_sequence(test_sequences, batch_first=True)
            test_pred = model.forward_temporal(test_padded, test_lengths).cpu().numpy()
            test_true = get_ground_truth(test_valid_names, target_data).numpy()
            test_ndcg = compute_ndcg_at_k(test_true, test_pred, k=50)
        else:
            test_ndcg = 0.0
            test_pred = np.array([])
            test_true = np.array([])
        
        del test_month_embeddings
        torch.cuda.empty_cache()
    
    print(f"\n  Final Results (Seed {seed}):")
    print(f"    Best Val NDCG@50: {best_val_ndcg:.4f}")
    print(f"    Test NDCG@50: {test_ndcg:.4f}")
    
    return {
        'seed': seed,
        'model': model,
        'best_val_ndcg': best_val_ndcg,
        'test_ndcg': test_ndcg,
        'test_pred': test_pred,
        'test_true': test_true,
        'test_names': test_valid_names
    }


print("V3 Training loop defined.")
print("  CRITICAL FIX: GCN is now TRAINED end-to-end!")
print("  Gradients flow: Loss -> FC -> Attention -> RNN -> GCN")

## 11. Train Ensemble

In [ ]:
print("\n" + "="*60)
print("STARTING V3 ENSEMBLE TRAINING (END-TO-END GCN)")
print("="*60)
print(f"Training {len(ENSEMBLE_SEEDS)} models with seeds: {ENSEMBLE_SEEDS}")
print(f"\nCRITICAL FIXES IN V3:")
print(f"  1. GCN is now TRAINED (was using random weights in v2!)")
print(f"  2. Non-influencer features preserved as one-hot (not normalized)")
print(f"  3. Extended leaky indices to include likes features ({len(LEAKY_INDICES)} total)")
print(f"\nEXPECTED: NDCG@50 should be 0.70-0.74 (with trained GCN)")
print(f"Previous v2 (random GCN): ~0.55-0.65 NDCG@50")
print(f"Expected improvement: +0.10-0.15 from trained GCN")

ensemble_results = []

for seed in ENSEMBLE_SEEDS:
    result = train_single_model_v3(
        seed, converted_graphs, 
        train_influencers_list, val_influencers_list, test_influencers_list,
        all_graphs_data
    )
    ensemble_results.append(result)
    
    # Clear GPU memory between models
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("ENSEMBLE TRAINING COMPLETE")
print("="*60)

## 12. Ensemble Prediction

In [ ]:
print("\nComputing ensemble predictions...")

# Average predictions across all models
ensemble_predictions = np.zeros_like(ensemble_results[0]['test_pred'])
for result in ensemble_results:
    ensemble_predictions += result['test_pred']
ensemble_predictions /= len(ensemble_results)

# Compute ensemble NDCG@50
test_true = ensemble_results[0]['test_true']
ensemble_ndcg = compute_ndcg_at_k(test_true, ensemble_predictions, k=50)

print(f"\n" + "="*60)
print("FINAL RESULTS (V3: END-TO-END GCN TRAINING)")
print("="*60)

print(f"\nIndividual Model Results:")
for result in ensemble_results:
    print(f"  Seed {result['seed']}: Val={result['best_val_ndcg']:.4f}, Test={result['test_ndcg']:.4f}")

avg_test_ndcg = np.mean([r['test_ndcg'] for r in ensemble_results])
std_test_ndcg = np.std([r['test_ndcg'] for r in ensemble_results])

print(f"\nSingle Model Stats:")
print(f"  Mean NDCG@50: {avg_test_ndcg:.4f} (+/- {std_test_ndcg:.4f})")

print(f"\nEnsemble Results:")
print(f"  Ensemble NDCG@50: {ensemble_ndcg:.4f}")

print(f"\nComparison to previous versions:")
v1_ndcg = 0.6951
v2_ndcg_expected = 0.60  # Approximate (random GCN weights)
improvement_v1 = ensemble_ndcg - v1_ndcg
improvement_v2 = ensemble_ndcg - v2_ndcg_expected
print(f"  v1 NDCG@50 (self-loops): {v1_ndcg:.4f}")
print(f"  v2 NDCG@50 (random GCN): ~{v2_ndcg_expected:.4f}")
print(f"  v3 NDCG@50 (trained GCN): {ensemble_ndcg:.4f}")
print(f"  Improvement over v1: {improvement_v1:+.4f}")
print(f"  Improvement over v2: {improvement_v2:+.4f}")

print(f"\n" + "="*60)
if ensemble_ndcg >= 0.72:
    print(f"TARGET ACHIEVED! NDCG@50 = {ensemble_ndcg:.4f} >= 0.72")
    print(f"Paper target: 0.720, Achieved: {ensemble_ndcg:.4f}")
elif ensemble_ndcg >= 0.70:
    print(f"GOOD PROGRESS! NDCG@50 = {ensemble_ndcg:.4f} >= 0.70")
    print("Close to paper target (0.720)")
elif ensemble_ndcg >= 0.65:
    print(f"IMPROVEMENT! NDCG@50 = {ensemble_ndcg:.4f} >= 0.65")
    print("Better than v2, consider further optimization")
else:
    print(f"NDCG@50 = {ensemble_ndcg:.4f}")
    print("Check for remaining issues")
print("="*60)

## 13. Save Results

In [ ]:
# Save ensemble results
results_summary = {
    'version': 'v3_end_to_end_gcn_training',
    'ensemble_ndcg': float(ensemble_ndcg),
    'individual_results': [
        {
            'seed': r['seed'],
            'val_ndcg': float(r['best_val_ndcg']),
            'test_ndcg': float(r['test_ndcg'])
        }
        for r in ensemble_results
    ],
    'avg_test_ndcg': float(avg_test_ndcg),
    'std_test_ndcg': float(std_test_ndcg),
    'config': {
        'input_dim': INPUT_DIM,
        'gnn_hidden': GNN_HIDDEN,
        'gnn_out': GNN_OUT,
        'rnn_hidden': RNN_HIDDEN,
        'dropout': DROPOUT,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'list_size': LIST_SIZE,
        'leaky_indices_zeroed': LEAKY_INDICES,
        'num_leaky_features': len(LEAKY_INDICES),
        'uses_actual_graph_structure': True,
        'gcn_trained_end_to_end': True,  # CRITICAL FIX!
        'non_influencer_features_normalized': False  # CRITICAL FIX!
    },
    'fixes_applied': [
        'GCN trained end-to-end (was using random weights)',
        'Non-influencer features preserved as one-hot (not normalized)',
        'Extended leaky indices to include likes-based features',
        'Removed unused model component (internal GCN that was never called)',
        'Removed unused function parameter (embedding_dim)'
    ],
    'test_predictions': ensemble_predictions.tolist(),
    'test_ground_truth': test_true.tolist(),
    'test_influencers': ensemble_results[0]['test_names']
}

# Save to file
import json
with open('v3_end_to_end_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to v3_end_to_end_results.json")

## 14. Analysis and Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot predicted vs actual rankings
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Scatter plot
ax1 = axes[0]
ax1.scatter(test_true, ensemble_predictions, alpha=0.5, s=10)
ax1.set_xlabel('True Engagement Rate')
ax1.set_ylabel('Predicted Score')
ax1.set_title('Predicted vs True Engagement')
ax1.grid(True, alpha=0.3)

# Ranking comparison (top 50)
ax2 = axes[1]
true_ranks = np.argsort(test_true)[::-1][:50]
pred_ranks = np.argsort(ensemble_predictions)[::-1][:50]

# How many of top 50 predicted are in top 50 true?
overlap = len(set(true_ranks) & set(pred_ranks))
ax2.bar(['Top 50 Overlap'], [overlap], color='green', alpha=0.7)
ax2.set_ylabel('Number of Influencers')
ax2.set_title(f'Top 50 Ranking Overlap ({overlap}/50 = {100*overlap/50:.1f}%)')
ax2.set_ylim([0, 50])
ax2.axhline(y=25, color='red', linestyle='--', label='Random baseline (50%)')
ax2.legend()

# Version comparison
ax3 = axes[2]
versions = ['v1\n(self-loops)', 'v2\n(random GCN)', 'v3\n(trained GCN)', 'Paper\nTarget']
ndcgs = [0.6951, 0.60, ensemble_ndcg, 0.720]
colors = ['gray', 'orange', 'green', 'blue']
ax3.bar(versions, ndcgs, color=colors, alpha=0.7)
ax3.set_ylabel('NDCG@50')
ax3.set_title('Version Comparison')
ax3.set_ylim([0.5, 0.8])
for i, v in enumerate(ndcgs):
    ax3.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.suptitle(f'V3 Ensemble NDCG@50: {ensemble_ndcg:.4f} (End-to-End GCN Training)', fontsize=14)
plt.tight_layout()
plt.savefig('v3_end_to_end_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 50 ranking overlap: {overlap}/50 ({100*overlap/50:.1f}%)")
print("Analysis saved to v3_end_to_end_analysis.png")

## 15. Conclusion

In [ ]:
print("\n" + "="*60)
print("SUMMARY - VERSION 3 (END-TO-END GCN TRAINING)")
print("="*60)

print("\n1. CRITICAL FIX #1: GCN NOW TRAINED")
print("   - v2 BUG: External GCN computed embeddings with RANDOM weights")
print("   - v2 BUG: Model's internal GCN was never used")
print("   - v3 FIX: GCN is called during training, gradients flow through it")
print(f"   - Expected improvement: +0.10 to +0.15 NDCG@50")

print("\n2. CRITICAL FIX #2: NON-INFLUENCER FEATURES PRESERVED")
print("   - v2 BUG: One-hot features normalized with influencer scaler")
print("     Example: hashtag [0,1,0,0,...] became [-3.75,-3.11,...]")
print("   - v3 FIX: Non-influencer features kept as one-hot encoding")
print(f"   - Expected improvement: +0.02 to +0.05 NDCG@50")

print("\n3. EXTENDED LEAKY INDICES")
print(f"   - v2: Zeroed {8} features (engagement-based only)")
print(f"   - v3: Zeroed {len(LEAKY_INDICES)} features (engagement + likes-based)")
print("   - Added: likes_trend, likes_variance, likes_consistency, etc.")
print(f"   - This ensures honest evaluation (no likes correlation leakage)")

print("\n4. PAPER-MATCHING ARCHITECTURE")
print("   - Simple 2-layer GCN (not GAT)")
print("   - Single-layer GRU (not 4-layer bidirectional)")
print("   - Simple attention (not multi-head)")
print("   - 2 FC layers (not 3)")
print("   - 0.5 dropout (paper's value)")
print("   - 0.001 learning rate (paper's value)")

print("\n5. RESULTS")
print(f"   - Single Model Mean: {avg_test_ndcg:.4f} (+/- {std_test_ndcg:.4f})")
print(f"   - Ensemble NDCG@50: {ensemble_ndcg:.4f}")
print(f"   - v1 (self-loops): {v1_ndcg:.4f}")
print(f"   - v2 (random GCN): ~{v2_ndcg_expected:.4f}")
print(f"   - Improvement over v1: {improvement_v1:+.4f}")
print(f"   - Improvement over v2: {improvement_v2:+.4f}")

if ensemble_ndcg >= 0.72:
    print(f"\n TARGET ACHIEVED! {ensemble_ndcg:.4f} >= 0.72")
    print(f"   Paper target: 0.720, Achieved: {ensemble_ndcg:.4f}")
elif ensemble_ndcg >= 0.70:
    print(f"\n GOOD PROGRESS! {ensemble_ndcg:.4f} >= 0.70")
    print("   Close to paper target (0.720)")
elif ensemble_ndcg >= 0.65:
    print(f"\n IMPROVEMENT! {ensemble_ndcg:.4f} >= 0.65")
    print("   Better than v2 (random GCN weights)")
else:
    print(f"\n NDCG@50 = {ensemble_ndcg:.4f}")
    print("   Consider further optimization")

print("\n" + "="*60)